# План
1. Создать онтологию
2. Создать промпт
3. Построить граф на тексте
4. Код RAG
5. Промпт RAG
6. 

In [ ]:
# 0 утсановка ключей в env

In [26]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv("../.env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [27]:
# 1 стирание базы полное до нуля

In [28]:
def step_1_reset_db():
    !docker stop neo4j_travel || true
    !docker rm neo4j_travel || true
    !docker volume rm neo4j_travel_data || true
    
    !docker run -d --name neo4j_travel \
      -p 7475:7474 -p 7688:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel_data:/data \
      neo4j:5
    
    # !docker ps --filter name=neo4j_travel   
    print("Did reset DB")
    return None


In [29]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [39]:
def step_2_prompt_to_abox(msg_as_JSON) -> str:
    import os
    import json
    import requests
    import time

    PROMPT_PATH = "prompt_extractor.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY (export DEEPSEEK_API_KEY=...).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-chat

    # --- load prompt template + ontology ---
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        prompt_template = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    # --- normalize message JSON string ---
    if isinstance(msg_as_JSON, (dict, list)):
        message_json_string = json.dumps(msg_as_JSON, ensure_ascii=False)
    else:
        message_json_string = str(msg_as_JSON).strip()

    # validate that it's JSON (LLM expects JSON string with keys post_id/post_url/author/date/text)
    try:
        json.loads(message_json_string)
    except Exception as e:
        raise ValueError(f"msg_as_JSON must be a valid JSON string (or dict). Parse error: {e}") from e

    # --- stitch prompt ---
    prompt = (
        prompt_template
        .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
        .replace("{MESSAGE_JSON_STRING_HERE}", message_json_string)
    )

    # sanity check for unresolved placeholders
    unresolved = [p for p in ["{RDF_ONTOLOGY_HERE}", "{MESSAGE_JSON_STRING_HERE}"] if p in prompt]
    if unresolved:
        raise RuntimeError(f"Unresolved placeholders in prompt: {unresolved}")

    # --- call DeepSeek ---
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            # system можно оставить очень коротким, промпт уже строгий
            {"role": "system", "content": "Return ONLY valid JSON object. No markdown. No code fences."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    print(f"DeepSeek API latency: {elapsed:.2f}s")
    resp.raise_for_status()
    data = resp.json()

    api_finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]

    emoji = "🟢" if api_finish != "length" else "🔴"
    print(f"{emoji} API finish_reason = {api_finish}")

    # optional: validate JSON (hard fail if model returned junk)
    try:
        json.loads(content)
    except Exception as e:
        raise ValueError(f"Model did not return valid JSON. Error: {e}\nRaw content:\n{content[:2000]}") from e

    print(content)  # preview
    return content


In [31]:
# 3 JSON → Cypher генерация

In [32]:
from typing import List

def step_4_abox_to_cypher(LLM_answer: str) -> List[str]:
    import json
    import re

    # =========================
    # Helpers
    # =========================

    def strip_code_fences(s: str) -> str:
        s = (s or "").strip()
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s)
            s = re.sub(r"\s*```$", "", s)
        return s.strip()

    def neo4j_local_name(name: str) -> str:
        if not name:
            return ""
        # ex:entityName -> entityName
        if ":" in name:
            return name.split(":")[-1]
        if "#" in name:
            return name.split("#")[-1]
        if "/" in name:
            return name.rsplit("/", 1)[-1]
        return name

    def neo4j_safe_ident(name: str) -> str:
        """
        Neo4j-safe identifier for labels / rel-types / property keys.
        Keep ASCII letters/digits/_ ; everything else -> _
        """
        n = neo4j_local_name(name)
        n = re.sub(r"[^A-Za-z0-9_]", "_", n)
        if not n:
            return "X"
        if n[0].isdigit():
            n = "_" + n
        return n

    def contains_cyrillic(s: str) -> bool:
        return bool(re.search(r"[А-Яа-яЁё]", s or ""))

    def cypher_literal(v):
        if v is None:
            return "null"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float)):
            return str(v)

        # strings
        s = str(v)
        s = s.replace("\\", "\\\\").replace("'", "\\'")
        s = s.replace("\n", "\\n").replace("\r", "\\r")
        return f"'{s}'"

    # =========================
    # Parse JSON from LLM
    # =========================

    raw = strip_code_fences(LLM_answer)
    root = json.loads(raw)

    payload = root.get("payload", {}) or {}
    nodes = payload.get("nodes", []) or []
    rels = payload.get("relationships", []) or []

    cypher_statements: List[str] = []

    # =========================
    # Constraints (unique key per label)
    # =========================

    labels = sorted({
        neo4j_safe_ident(n.get("label", ""))
        for n in nodes
        if n.get("label")
    })

    for label in labels:
        cypher_statements.append(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
        )

    # =========================
    # Nodes (MERGE by key, then SET props)
    # =========================

    for node in nodes:
        label_raw = node.get("label", "")
        key = node.get("key")

        if not label_raw or key is None:
            continue

        label = neo4j_safe_ident(label_raw)
        props = node.get("properties", {}) or {}

        set_parts = []
        for k, v in props.items():
            prop_key = neo4j_safe_ident(k)

            if isinstance(v, str) and contains_cyrillic(v):
                print(f"⚠ WARNING: Cyrillic detected in property '{prop_key}' → '{v}'")

            set_parts.append(f"n.{prop_key}={cypher_literal(v)}")

        set_clause = (" SET " + ", ".join(set_parts)) if set_parts else ""
        cypher_statements.append(
            f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
        )

    # =========================
    # Relationships
    # =========================

    for rel in rels:
        f = rel.get("from", {}) or {}
        t = rel.get("to", {}) or {}

        fl_raw = f.get("label", "")
        fk = f.get("key")
        tl_raw = t.get("label", "")
        tk = t.get("key")
        rt_raw = rel.get("type", "")

        if not fl_raw or not tl_raw or not rt_raw or fk is None or tk is None:
            continue

        fl = neo4j_safe_ident(fl_raw)
        tl = neo4j_safe_ident(tl_raw)
        rt = neo4j_safe_ident(rt_raw)

        cypher_statements.append(
            f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
            f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
            f"MERGE (a)-[:{rt}]->(b)"
        )

    # =========================
    # Output
    # =========================

    print("LLM finish_reason:", root.get("finish_reason"))
    print("LLM reason:", root.get("reason"))
    print(f"Prepared {len(cypher_statements)} Cypher statements.\n")

    return cypher_statements

In [33]:
# 4 Сохранение Cypher в чекпоинт

In [34]:
def step_5_save_cypher(cypher_statements: List[str]):
    import os
    import re
    
    ckpt_dir = "cypher_checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    existing = []
    for name in os.listdir(ckpt_dir):
        m = re.match(r"^cypher_(\d+)\.txt$", name)
        if m:
            existing.append(int(m.group(1)))
    
    next_num = max(existing, default=0) + 1
    out_path = os.path.join(ckpt_dir, f"cypher_{next_num}.txt")
    
    with open(out_path, "w", encoding="utf-8") as f:
        for stmt in cypher_statements:
            f.write(stmt)
            f.write(";")
    
    print(f"Saved {len(cypher_statements)} statements to {out_path}")
    return None


In [35]:
# 5 исполнение Cypher (каждая строка автономна)

In [36]:
def step_6_execute_cypher(cypher_statements: List[str]):
    from neo4j import GraphDatabase
    
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"
    
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    
    try:
        with driver.session(database=DATABASE) as session:
            for i, stmt in enumerate(cypher_statements, 1):
                session.run(stmt).consume()
        print(f"✅ Executed {len(cypher_statements)} statements.")
    finally:
        driver.close()    
    return None


In [37]:
# 7 запуск пайплайна

In [40]:
step_1_reset_db()

import json
import subprocess


with open("messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)
messages = list(map(lambda x: json.dumps(x, ensure_ascii=False), messages[:1])) # берем первое и только его


for msg in messages:
    print(msg)
    abox_json = step_2_prompt_to_abox(msg) 
    cypher_statements = step_4_abox_to_cypher(abox_json)
    step_5_save_cypher(cypher_statements)
    step_6_execute_cypher(cypher_statements)

subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel
neo4j_travel
neo4j_travel_data
6d4d2d6e84585152e0db61cbd3e5d708286b03095b39b3141bb9cdca38bb57e6
Did reset DB
{"post_id": 77777777, "post_url": "https://vk.com", "author": "IVAN_SHARGANOV", "date": "11 июн 2013, 09:05", "text": "Elon Musk is the CEO of SpaceX. SpaceX was founded in 2002. Elon Musk was born in South Africa. Elon Musk’s email is elon@spacex.com."}
DeepSeek API latency: 77.72s
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "https://example.org/elon#",
    "nodes": [
      {
        "label": "Person",
        "key": "person_elon_musk",
        "properties": {
          "entityName": "Elon Musk",
          "email": "elon@spacex.com"
        }
      },
      {
        "label": "Company",
        "key": "company_spacex",
        "properties": {
          "entityName": "SpaceX",
          "foundedInYear": 2002
        }
      },
      {
        "label": "Country",
        "key": "country_south_africa",
        "properties": {
          "entityN

NameError: name 'subprocess' is not defined

In [41]:
#  RAG

In [63]:
def optimise_retrieval_prompt_with_gold(
    gold_path: str = "gold_retrieval.jsonl",
    prompt_retrieval_path: str = "prompt_retrieval.txt",
    prompt_optimiser_path: str = "prompt_retr_optimiser.txt",
    versions_dir: str = "prompt_versions",
    holdout_index: int = 0,
    temperature: float = 0.0,
) -> None:
    """
    Iterative prompt optimization for retrieval prompt using gold (question -> gold_cypher).
    Leaves one example (holdout_index) for metric evaluation, trains on the rest.

    Versioning:
      - before each overwrite of prompt_retrieval.txt, a snapshot is saved into versions_dir/
        with timestamp + step index + case id.
    """

    from neo4j import GraphDatabase
    import os, json, requests
    from pathlib import Path
    from datetime import datetime

    # -----------------------------
    # Config
    # -----------------------------
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("DEEPSEEK_API_KEY is not set (load .env before calling).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}

    # -----------------------------
    # File I/O
    # -----------------------------
    gold_file = Path(gold_path)
    pr_file = Path(prompt_retrieval_path)
    opt_file = Path(prompt_optimiser_path)
    ver_dir = Path(versions_dir)

    if not gold_file.exists():
        raise FileNotFoundError(gold_file.resolve())
    if not pr_file.exists():
        raise FileNotFoundError(pr_file.resolve())
    if not opt_file.exists():
        raise FileNotFoundError(opt_file.resolve())

    ver_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------
    # Helpers
    # -----------------------------
    def call_llm(prompt_text: str, temp: float = 0.0) -> str:
        payload = {
            "model": MODEL,
            "messages": [{"role": "user", "content": prompt_text}],
            "temperature": temp,
        }
        resp = requests.post(url, headers=headers, json=payload, timeout=180)
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]

    def run_cypher(cypher: str):
        if not cypher.strip():
            return {"ok": False, "error": "empty cypher", "rows": []}
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        try:
            with driver.session(database=DATABASE) as session:
                rows = session.run(cypher).data()
                return {"ok": True, "error": "", "rows": rows}
        except Exception as e:
            return {"ok": False, "error": str(e), "rows": []}
        finally:
            driver.close()

    def canonical_rows(rows):
        try:
            norm = [json.loads(json.dumps(r, sort_keys=True, ensure_ascii=False)) for r in rows]
            return sorted(norm, key=lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False))
        except Exception:
            return rows

    def get_schema_text() -> str:
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        try:
            with driver.session(database=DATABASE) as session:
                labels = session.run(
                    "CALL db.labels() YIELD label RETURN collect(label) AS labels"
                ).single()["labels"]

                rel_types = session.run(
                    "CALL db.relationshipTypes() YIELD relationshipType "
                    "RETURN collect(relationshipType) AS relTypes"
                ).single()["relTypes"]

                label_props = {}
                for label in labels:
                    result = session.run(f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1").single()
                    label_props[label] = result["keys"] if result else []
        finally:
            driver.close()

        return (
            f"Available node labels: {labels}\n"
            f"Available relationship types: {rel_types}\n"
            f"Node properties by label: {json.dumps(label_props, ensure_ascii=False)}\n"
        )

    def save_snapshot(text: str, step: int, case_id: str, kind: str = "before") -> Path:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_id = "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in (case_id or "no_id"))
        fname = f"{ts}__step{step:03d}__{kind}__{safe_id}.txt"
        path = ver_dir / fname
        path.write_text(text, encoding="utf-8")
        return path

    # -----------------------------
    # Load gold cases (JSONL)
    # -----------------------------
    gold_cases = []
    for line in gold_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            gold_cases.append(json.loads(line))

    if len(gold_cases) < 2:
        raise RuntimeError("Need at least 2 gold cases (1 holdout + >=1 train).")

    if holdout_index < 0 or holdout_index >= len(gold_cases):
        raise ValueError("holdout_index out of range.")

    holdout = gold_cases[holdout_index]
    train = [c for i, c in enumerate(gold_cases) if i != holdout_index]

    schema_text = get_schema_text()

    # -----------------------------
    # Training loop: one pass over dataset
    # -----------------------------
    for step, case in enumerate(train, start=1):
        case_id = case.get("id", f"case_{step}")
        current_prompt = pr_file.read_text(encoding="utf-8")
        optimiser_prompt = opt_file.read_text(encoding="utf-8")

        # 1) generate model cypher
        retrieval_prompt = (
            current_prompt
            .replace("{SCHEMA_TEXT}", schema_text)
            .replace("{USER_QUERY}", case["question"])
        )
        model_out = call_llm(retrieval_prompt, temp=0.0)

        try:
            model_json = json.loads(model_out)
            model_cypher = (model_json.get("cypher") or "").strip()
        except Exception:
            model_cypher = ""

        # 2) run gold + model
        gold_cypher = (case.get("gold_cypher") or "").strip()
        gold_exec = run_cypher(gold_cypher)
        model_exec = run_cypher(model_cypher)

        training_case = {
            "question": case["question"],
            "gold_cypher": gold_cypher,
            "model_cypher": model_cypher,
            "model_error": model_exec["error"],
            "gold_result": gold_exec["rows"][:50],
            "model_result": model_exec["rows"][:50],
        }

        # 3) ask optimiser to produce NEW full prompt text
        opt_in = (
            optimiser_prompt
            .replace("{CURRENT_PROMPT}", current_prompt)
            .replace("{SCHEMA_TEXT}", schema_text)
            .replace("{TRAINING_CASE_JSON}", json.dumps(training_case, ensure_ascii=False))
        )
        new_prompt_text = call_llm(opt_in, temp=temperature).strip()
        print("🔥")
        print(new_prompt_text)
        
        # validate placeholders (soft fail -> keep current prompt)
        if "{SCHEMA_TEXT}" not in new_prompt_text or "{USER_QUERY}" not in new_prompt_text:
            print("⚠ Optimiser returned invalid prompt (missing placeholders). Keeping CURRENT prompt unchanged.")
            new_prompt_text = current_prompt

        # 4) versioning: save snapshot BEFORE overwrite, then write, then save AFTER
        before_path = save_snapshot(current_prompt, step=step, case_id=case_id, kind="before")
        pr_file.write_text(new_prompt_text, encoding="utf-8")
        after_path = save_snapshot(new_prompt_text, step=step, case_id=case_id, kind="after")

        print(f"✅ Step {step}/{len(train)} case={case_id} | snapshots:")
        print(f"   - {before_path}")
        print(f"   - {after_path}")

    # -----------------------------
    # Holdout evaluation (simple metric)
    # -----------------------------
    final_prompt = pr_file.read_text(encoding="utf-8")
    holdout_prompt = (
        final_prompt
        .replace("{SCHEMA_TEXT}", schema_text)
        .replace("{USER_QUERY}", holdout["question"])
    )
    holdout_out = call_llm(holdout_prompt, temp=0.0)

    try:
        holdout_json = json.loads(holdout_out)
        holdout_model_cypher = (holdout_json.get("cypher") or "").strip()
    except Exception:
        holdout_model_cypher = ""

    gold_exec = run_cypher((holdout.get("gold_cypher") or "").strip())
    model_exec = run_cypher(holdout_model_cypher)

    gold_rows = canonical_rows(gold_exec["rows"])
    model_rows = canonical_rows(model_exec["rows"])

    exact_result_match = (gold_exec["ok"] and model_exec["ok"] and gold_rows == model_rows)

    print("\n--- Holdout evaluation ---")
    print("Question:", holdout["question"])
    print("Gold ok:", gold_exec["ok"], "Model ok:", model_exec["ok"])
    print("Exact result match:", exact_result_match)
    if not model_exec["ok"]:
        print("Model error:", model_exec["error"])
    print("\nHoldout gold cypher:\n", holdout.get("gold_cypher", ""))
    print("\nHoldout model cypher:\n", holdout_model_cypher)

In [50]:
rag_answer("SpaceX's CEO?")


--- Injected Schema ---
Available node labels: ['Company', 'Country', 'Person']
Available relationship types: ['CEO_OF', 'BORN_IN']
Node properties by label: {"Company": ["entityName", "foundedInYear", "key"], "Country": ["entityName", "key"], "Person": ["entityName", "email", "key"]}


--- Generated Cypher ---
MATCH (c:Company {entityName: 'SpaceX'}) OPTIONAL MATCH (p:Person)-[:CEO_OF]-(c) RETURN p.entityName AS ceo_name LIMIT 1

--- Retrieved Facts ---
[{"ceo_name": "Elon Musk"}]

--- Final Answer ---
SpaceX's CEO is Elon Musk.


In [64]:
optimise_retrieval_prompt_with_gold()

🔥
You are a Neo4j Cypher query generator.

You will receive:
1) A database schema description (node labels, relationship types, and example property keys).
2) A user question.

Your task:
Generate ONE read-only Cypher query that retrieves the facts necessary to answer the question.

STRICT RULES (MUST FOLLOW)

1) You MUST use ONLY:
   - node labels present in the provided schema,
   - relationship types present in the provided schema,
   - property keys present in the provided schema.
   Do NOT invent anything not explicitly listed.

2) Allowed Cypher clauses:
   MATCH, OPTIONAL MATCH, WHERE, WITH, RETURN,
   DISTINCT, ORDER BY, LIMIT,
   CASE,
   aggregate functions (count, sum, avg, min, max, collect).

3) Forbidden clauses:
   CREATE, MERGE, SET, DELETE, REMOVE, CALL,
   LOAD CSV, APOC procedures, or any write operation.

4) Always include LIMIT:
   - Use LIMIT 50 for list-style outputs.
   - Use LIMIT 1 for aggregated/single-row outputs.

5) If the question asks about frequency, pr